In [1]:
import pandas as pd
import numpy as np

# =========================
# 1. Load data
# =========================
df = pd.read_csv("data1C/model_option_A.csv")

# Parse dates
df["date"] = pd.to_datetime(df["date"])
if "next_date" in df.columns:
    df["next_date"] = pd.to_datetime(df["next_date"])

# Sort per user and date
df = df.sort_values(["id", "date"]).reset_index(drop=True)

# =========================
# 2. Keep only valid next-day transitions
#    (important for predicting next-day mood)
# =========================
if "next_date" in df.columns:
    df = df[df["next_date"] == (df["date"] + pd.Timedelta(days=1))].copy()
else:
    # fallback if next_date is not present
    df["next_date_check"] = df.groupby("id")["date"].shift(-1)
    df["mood_next_day"] = df.groupby("id")["mood"].shift(-1)
    df = df[df["next_date_check"] == (df["date"] + pd.Timedelta(days=1))].copy()
    df = df.drop(columns=["next_date_check"])

# =========================
# 3. Define current-day feature columns
#    These stay in the dataset
# =========================
feature_cols = [
    "activity",
    "appCat.builtin",
    "appCat.communication",
    "appCat.entertainment",
    "appCat.finance",
    "appCat.game",
    "appCat.office",
    "appCat.other",
    "appCat.social",
    "appCat.travel",
    "appCat.unknown",
    "appCat.utilities",
    "appCat.weather",
    "call",
    "sms",
    "screen",
    "mood"
]

# =========================
# 4. Create history-based features
#    We use previous days only for history
# =========================
new_features = {}

for col in feature_cols:
    g = df.groupby("id")[col]

    shifted = g.shift(1)  # yesterday
    shifted2 = g.shift(2) # two days ago

    new_features[f"{col}_lag1"] = shifted
    new_features[f"{col}_lag2"] = shifted2

    # 3-day history (previous days only)
    new_features[f"{col}_mean_3d_hist"] = shifted.rolling(3, min_periods=1).mean()
    new_features[f"{col}_std_3d_hist"]  = shifted.rolling(3, min_periods=2).std()
    new_features[f"{col}_sum_3d_hist"]  = shifted.rolling(3, min_periods=1).sum()

    # 5-day history (previous days only)
    new_features[f"{col}_mean_5d_hist"] = shifted.rolling(5, min_periods=1).mean()

history_df = pd.DataFrame(new_features, index=df.index)
df = pd.concat([df, history_df], axis=1)

# =========================
# 5. Add a few simple change/trend features
# =========================
df["mood_change_today_vs_yesterday"] = df["mood"] - df["mood_lag1"]
df["screen_change_today_vs_yesterday"] = df["screen"] - df["screen_lag1"]
df["activity_change_today_vs_yesterday"] = df["activity"] - df["activity_lag1"]

# Optional calendar features
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# =========================
# 6. Keep only rows with enough history
#    Here: at least 2 previous days + 3-day history available
# =========================
required_history = [
    "mood_lag1",
    "mood_lag2",
    "mood_mean_3d_hist"
]

df_final = df.dropna(subset=required_history + ["mood_next_day"]).copy()

# =========================
# 7. Drop helper columns that should not be used as predictors
# =========================
cols_to_drop = [c for c in ["num_nonzero", "is_fake", "next_date"] if c in df_final.columns]
df_final = df_final.drop(columns=cols_to_drop)

# =========================
# 8. Save final Task 1C dataset
# =========================
df_final.to_csv("data1C/final_data_1C.csv", index=False)

print("Saved file: final_data_1C.csv")
print("Shape:", df_final.shape)
print(df_final.head())

Saved file: final_data_1C.csv
Shape: (1162, 127)
        id       date  activity  appCat.builtin  appCat.communication  \
2  AS14.01 2014-03-22  0.236880         731.429              4962.918   
3  AS14.01 2014-03-23  0.142741        1286.246              5237.319   
4  AS14.01 2014-03-24  0.078961         866.956              9270.629   
5  AS14.01 2014-03-25  0.098374        1032.768             10276.751   
6  AS14.01 2014-03-26  0.101308        1167.497              8988.753   

   appCat.entertainment  appCat.finance  appCat.game  appCat.office  \
2                93.324          21.076          0.0           0.00   
3                94.346          43.403          0.0           0.00   
4               976.971          34.106          0.0           3.01   
5                68.206          43.054          0.0           0.00   
6               910.479          52.331          0.0           0.00   

   appCat.other  ...  mood_lag2  mood_mean_3d_hist  mood_std_3d_hist  \
2        98.1